In [1]:
#import geopandas as gpd
#import matplotlib.pyplot as plt
#import plotly.express as px
import pandas as pd 
#from shapely import wkt
#import matplotlib.pyplot as plt
#import matplotlib.cm as cm
import numpy as np

In [2]:
#gdf = pd.DataFrame(gpd.read_file("recintos.gpkg"))
votos = pd.DataFrame(pd.read_excel("data/input/votos_2da.xlsx"))
df = (votos.
           #pipe(lambda x: x[x["NombrePais"]=="BOLIVIA"]).
           pipe(lambda x: x[x["Descripcion"]=="PRESIDENTE"]).
           #merge(gdf,how="left",left_on=["CodigoLocalidad","CodigoRecinto"], right_on = ["idloc","recinto_codigo"]).
           rename(columns= {"PDC":"v_PDC","LIBRE":"v_LIBRE"}).
           assign( c_ut = lambda x: x['CodigoDepartamento'].apply(lambda x: f"{x:02d}") +
                                    x['CodigoProvincia'].apply(lambda x: f"{x:02d}")+
                                    x['CodigoSeccion'].apply(lambda x: f"{x:02d}"))
           )


df

,CodigoMesa,Descripcion,CodigoPais,NombrePais,CodigoDepartamento,NombreDepartamento,CodigoCircunscripcionU,CodigoCircunscripcionE,CodigoProvincia,NombreProvincia,...,v_LIBRE,VotoValido,VotoBlanco,VotoNuloDirecto,VotoNuloDeclinacion,TotalVotoNulo,VotoEmitido,VotoValidoReal,VotoEmitidoReal,c_ut
0,100014,PRESIDENTE,4,Alemania,61,Berlín,0,0,1,Berlín,...,79,93,2,4,0,4,99,93,99,610101
1,100024,PRESIDENTE,11,Argentina,10,Buenos Aires,0,0,1,Buenos Aires,...,48,93,1,1,0,1,95,93,95,100101
2,100034,PRESIDENTE,11,Argentina,10,Buenos Aires,0,0,1,Buenos Aires,...,52,86,0,6,0,6,92,86,92,100101
3,100044,PRESIDENTE,11,Argentina,10,Buenos Aires,0,0,1,Buenos Aires,...,29,87,0,5,0,5,92,87,92,100101
4,100054,PRESIDENTE,11,Argentina,10,Buenos Aires,0,0,1,Buenos Aires,...,42,85,1,11,0,11,97,85,97,100101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35248,9003901,PRESIDENTE,32,BOLIVIA,9,Pando,63,0,5,Federico Román,...,67,163,3,7,0,7,173,163,173,090503
35249,9003911,PRESIDENTE,32,BOLIVIA,9,Pando,63,0,5,Federico Román,...,44,129,0,2,0,2,131,129,131,090503
35250,9003921,PRESIDENTE,32,BOLIVIA,9,Pando,63,0,5,Federico Román,...,48,167,1,0,0,0,168,167,168,090503
35251,9003931,PRESIDENTE,32,BOLIVIA,9,Pando,63,0,5,Federico Román,...,7,31,0,3,0,3,34,31,34,090503


In [3]:
df.groupby(["NombreDepartamento","CodigoDepartamento"]).agg({"CodigoMesa":"count"})

,,CodigoMesa
NombreDepartamento,CodigoDepartamento,
Acre,79,1
Andalucia,30,18
Antofagasta,38,69
Arica,37,14
Atacama,94,4
...,...,...
Tarija,6,1814
Texas,86,3
Tucumán,93,2


In [4]:
df.filter(like = "v_").sum().apply(lambda x: "{:,}".format(x))

v_PDC      3,512,264
v_LIBRE    2,882,742
dtype: object

In [5]:
import geopandas as gpd
gdf_dep = gpd.read_file("shapes/departamentos/a__1_Departamento2.shx", encoding='latin1')
gdf_dep.to_file("data/output/dep.geojson", driver="GeoJSON")

c:\Users\hrodrigo\AppData\Local\anaconda3\Lib\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


In [3]:
import pandas as pd
import geopandas as gpd


voto_cols = [col for col in df.columns if col.startswith("v_")]


resumen = (
    df.
    groupby(["NombreMunicipio","c_ut"])[voto_cols].sum().reset_index().
    assign( ganador = lambda x:x[voto_cols].idxmax(axis=1) ).
    assign( Total = lambda x:x.loc[:, 'v_PDC':'v_LIBRE'].sum(1)).
    assign(**{col + '_porcentaje' : lambda x, c=col: x[c] / x['Total'] * 100 for col in voto_cols})
    
    )

gdf = gpd.read_file("shapes/municipios/municipios339.shp", encoding='latin1')  
import re

def fix_encoding(df):
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].apply(
            lambda x: x.encode('latin1').decode('utf-8').strip() if isinstance(x, str) else x
        )
    return df

gdf = fix_encoding(gdf)

cambios = {
    'AIOC Charagua Iyambae': 'Charagua',
    'AIOC Uru Chipaya' : 'Chipaya',
    'AIOC de Salinas' : 'Salinas de Garcí Mendoza',
    'Icla' : 'Villa Ricardo Mugia - Icla',
    'Pampagrande' : 'Pampa Grande',
    'Santa Cruz de La Sierra':'Santa Cruz de la Sierra',
    'Santiago de Huayllamarca' : 'Huayllamarca',
    'Sopachuy':'Sopachui',
    'TIOC Guaraní Chaqueño de Huacaya' : 'Huacaya',
    'AIOC Guaraní Kereimba Iyaambae' : 'Gutiérrez'
}


resumen["NombreMunicipio"] = resumen["NombreMunicipio"].apply(lambda x: x.strip()).replace(cambios)




gdf = gdf.rename(columns={"NOM_MUN": "NombreMunicipio"}) 

merged = gdf.merge(resumen, on = "c_ut", how="outer")
merged["NombreMunicipio"] = merged['NombreMunicipio'].fillna(merged["c_ut"])
merged = merged.drop(columns=["ogc_fid","id","objectid","shape_leng","shape_area"])

merged.to_file("municipios_votacion_segunda_vuelta.geojson", driver="GeoJSON")

In [9]:
# --- 2. FUNCIÓN DE COSTO PARA LA OPTIMIZACIÓN ---

def rc_objective(params):
    """
    Función objetivo: Minimiza el RMSE de la predicción a corto plazo.
    Parámetros: [RHO, REG_PARAM]
    """
    RHO, REG_PARAM = params
    N_RES = 300 # Fijo
    
    # Inicialización y Escalado de Pesos
    np.random.seed(42)
    W_in = np.random.uniform(-1, 1, (N_RES, 1)) 
    W_res = np.random.uniform(-1, 1, (N_RES, N_RES))
    
    spectral_radius = np.max(np.abs(np.linalg.eigvals(W_res)))
    # Aplicación del fundamento teórico: RHO controla la estabilidad
    W_res = W_res * (RHO / spectral_radius) 

    # FASE DE ACUMULACIÓN DE ESTADOS
    X_states = []
    r = np.zeros(N_RES) 
    ALPHA = 0.3 # Fijo
    
    # Procesar la secuencia de entrada (Accede a input_data global)
    for u in input_data:
        u_curr = u[0]
        r_new = (1 - ALPHA) * r + ALPHA * np.tanh(W_in[:, 0] * u_curr + W_res @ r)
        r = r_new
        X_states.append(np.concatenate(([1], r)))

    # Alinear y Entrenar (Accede a target_data global)
    X_train = np.array(X_states[:-1])
    # Corrección de alineación: Y_train debe tener la misma longitud que X_train
    Y_train = target_data[:-1] 
    
    readout = Ridge(alpha=REG_PARAM)
    readout.fit(X_train, Y_train)
    W_out = readout.coef_ 
    
    # --- PRONÓSTICO A CORTO PLAZO (Evaluación) ---
    
    N_EVAL = 100 # Evaluar la precisión sobre 100 pasos
    r_eval = r.copy() 
    predictions = []
    
    for i in range(N_EVAL):
        r_augmented = np.concatenate(([1], r_eval))
        predicted_state = W_out @ r_augmented
        predictions.append(predicted_state)
        
        # Retroalimentación (Autónoma)
        u_next = predicted_state[0]
        r_eval = (1 - ALPHA) * r_eval + ALPHA * np.tanh(W_in[:, 0] * u_next + W_res @ r_eval)
        
    predictions = np.array(predictions)
    
    # FUNCIÓN DE COSTO
    true_eval_data = true_trajectory_long[:N_EVAL]
    rmse = mean_squared_error(true_eval_data, predictions, squared=False)
    
    # Añadimos una penalización si RHO es muy bajo (memoria pobre), para guiar
    # la optimización hacia el límite de estabilidad (cercano a 1).
    stability_penalty = 0 if RHO > 0.6 else (0.6 - RHO) * 10
    
    return rmse + stability_penalty # Retorna el costo combinado